In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df.drop('customerID', axis=1, inplace=True)

print("Data loaded successfully!", df.shape)

Data loaded successfully! (7043, 20)


/tmp/ipykernel_6373/3557701100.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [ ]:
# feature 1 - average monthly spend
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)

# feature 2 - count how many services customer has subscribed
service_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

df['ServiceCount'] = df[service_cols].apply(lambda x: (x == 'Yes').sum(), axis=1)

# feature 3 - is customer on monthly contract (risky customer)
df['IsMonthly'] = (df['Contract'] == 'Month-to-month').astype(int)

print("New features created!")
print(df[['AvgMonthlySpend', 'ServiceCount', 'IsMonthly']].head())

New features created!
   AvgMonthlySpend  ServiceCount  IsMonthly
0        14.925000             1          1
1        53.985714             3          0
2        36.050000             3          1
3        40.016304             3          0
4        50.550000             1          1


In [ ]:
from sklearn.preprocessing import LabelEncoder

# copy the dataframe
df2 = df.copy()

# label encoding for binary columns (only 2 values)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'Churn']

le = LabelEncoder()
for col in binary_cols:
    df2[col] = le.fit_transform(df2[col])

# one hot encoding for columns with more than 2 values
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
              'OnlineBackup', 'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

df2 = pd.get_dummies(df2, columns=multi_cols)

print("Encoding done!")
print("Shape after encoding:", df2.shape)

Encoding done!
Shape after encoding: (7043, 44)


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# columns to scale
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'ServiceCount']

# standard scaler
ss = StandardScaler()
df_standard = df2.copy()
df_standard[scale_cols] = ss.fit_transform(df2[scale_cols])

# minmax scaler
mm = MinMaxScaler()
df_minmax = df2.copy()
df_minmax[scale_cols] = mm.fit_transform(df2[scale_cols])

print("Scaling done!")
print("Standard Scaler sample:")
print(df_standard[scale_cols].head(3))
print("MinMax Scaler sample:")
print(df_minmax[scale_cols].head(3))

Scaling done!
Standard Scaler sample:
     tenure  MonthlyCharges  TotalCharges  AvgMonthlySpend  ServiceCount
0 -1.277445       -1.160323     -0.994242        -0.757979     -1.052777
1  0.066327       -0.259629     -0.173244        -0.117801      0.031958
2 -1.236724       -0.362660     -0.959674        -0.411755      0.031958
MinMax Scaler sample:
     tenure  MonthlyCharges  TotalCharges  AvgMonthlySpend  ServiceCount
0  0.013889        0.115423      0.001275         0.004136      0.142857
1  0.472222        0.385075      0.215867         0.032272      0.428571
2  0.027778        0.354229      0.010310         0.019352      0.428571


In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# separate features and target
x = df_standard.drop('Churn', axis=1)
y = df_standard['Churn']

# method 1 - correlation based filtering
correlation = x.corrwith(y).abs().sort_values(ascending=False)
print("Top 10 features by correlation:")
print(correlation.head(10))

# method 2 - RFE using random forest
model = RandomForestClassifier(random_state=42)
rfe = RFE(model, n_features_to_select=10)
rfe.fit(x, y)

# show selected features
selected = x.columns[rfe.support_]
print("\nTop 10 features by RFE:")
print(list(selected))

Top 10 features by correlation:
IsMonthly                         0.405103
Contract_Month-to-month           0.405103
tenure                            0.352229
OnlineSecurity_No                 0.342637
TechSupport_No                    0.337281
InternetService_Fiber optic       0.308020
Contract_Two year                 0.302253
PaymentMethod_Electronic check    0.301919
OnlineBackup_No                   0.268005
DeviceProtection_No               0.252481
dtype: float64

Top 10 features by RFE:
['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'ServiceCount', 'IsMonthly', 'InternetService_Fiber optic', 'OnlineSecurity_No', 'TechSupport_No', 'Contract_Month-to-month']


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# check before balancing
print("Before balancing:")
print(y.value_counts())

# method 1 - SMOTE (create fake minority samples)
sm = SMOTE(random_state=42)
x_smote, y_smote = sm.fit_resample(x, y)
print("\nAfter SMOTE:")
print(y_smote.value_counts())

# method 2 - Random Undersampling (remove majority samples)
rus = RandomUnderSampler(random_state=42)
x_under, y_under = rus.fit_resample(x, y)
print("\nAfter Undersampling:")
print(y_under.value_counts())

Before balancing:
Churn
0    5174
1    1869
Name: count, dtype: int64

After SMOTE:
Churn
0    5174
1    5174
Name: count, dtype: int64

After Undersampling:
Churn
0    1869
1    1869
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# we will use SMOTE balanced data
# first split 70% train and 30% remaining
x_train, x_temp, y_train, y_temp = train_test_split(
    x_smote, y_smote, test_size=0.30, random_state=42, stratify=y_smote)

# split remaining 30% into 15% validation and 15% test
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Train size:", x_train.shape)
print("Validation size:", x_val.shape)
print("Test size:", x_test.shape)

Train size: (7243, 43)
Validation size: (1552, 43)
Test size: (1553, 43)
